# LLT model probe 9B retry

In [ ]:

import json, os, subprocess, sys, time, platform
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.22.1", "peft==0.21.2", "accelerate", "bitsandbytes", "datasets"], check=False)
WORKER = '"""One probe job in its own process. Usage: python probe_worker.py <job.json> <result.json>\njob: {tag, model, mode: lora|fullft|qlora, steps, batch, time_cap}"""\nimport gc, json, os, sys, time, traceback, statistics, io, logging\nos.environ["CUDA_VISIBLE_DEVICES"] = "0"\nos.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"\njob = json.load(open(sys.argv[1])); outp = sys.argv[2]\nR = {"tag": job["tag"], "model": job["model"], "mode": job["mode"]}\ndef save():\n    json.dump(R, open(outp, "w"), indent=1)\nimport torch, transformers, peft\nfrom transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig\nlogbuf = io.StringIO()\nh = logging.StreamHandler(logbuf); h.setLevel(logging.WARNING)\nlogging.getLogger("transformers").addHandler(h)\ndef fla_state():\n    s = {}\n    for m in ("fla", "causal_conv1d"):\n        try:\n            __import__(m); s[m] = True\n        except Exception:\n            s[m] = False\n    return s\nR["fla"] = fla_state()\ntry:\n    from datasets import load_dataset\n    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")\n    text = "\\n".join(t for t in ds["text"][:20000] if len(t.strip()) > 40)[:900000]\n    tok = AutoTokenizer.from_pretrained(job["model"])\n    ids = tok(text, return_tensors="pt").input_ids[0]\n    R["tokensAvailable"] = int(ids.numel())\n    T = 1024; B = job["batch"]; N = job["steps"]\n    need = B * T * N\n    assert ids.numel() >= need, "not enough tokens"\n    stream = ids[:need].view(N, B, T)\n    t0 = time.time()\n    if job["mode"] == "qlora":\n        qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,\n                                  bnb_4bit_compute_dtype=torch.float16)\n        kw = dict(quantization_config=qcfg, dtype=torch.float16, device_map="cuda:0")\n    else:\n        kw = dict(dtype=torch.float32, device_map="cuda:0")\n    model = AutoModelForCausalLM.from_pretrained(job["model"], **kw)\n    R["loadSec"] = round(time.time() - t0, 1)\n    R["loadedClass"] = type(model).__name__\n    names = [n for n, _ in model.named_modules()]\n    R["visionTowerLoaded"] = any(("visual" in n or "vision" in n) for n in names)\n    R["modelType"] = getattr(model.config, "model_type", None)\n    R["totalParamsBase"] = sum(p.numel() for p in model.parameters())\n    R["loadWarnings"] = logbuf.getvalue()[:1500]\n    print("loaded", R["loadedClass"], "vision", R["visionTowerLoaded"], flush=True)\n    save()\n\n    if job["mode"] == "qlora":\n        # prepare_model_for_kbit_training upcasts the 248k-vocab embedding/lm_head to fp32 (3.79 GiB each): OOM on a T4. Do the safe parts by hand.\n        for p in model.parameters():\n            p.requires_grad = False\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.enable_input_require_grads()\n    else:\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        model.enable_input_require_grads()\n    model.config.use_cache = False\n\n    if job["mode"] in ("lora", "qlora"):\n        from peft import LoraConfig, get_peft_model\n        ex = r".*(visual|vision).*" if R["visionTowerLoaded"] else None\n        cfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, target_modules="all-linear", exclude_modules=ex, task_type="CAUSAL_LM")\n        model = get_peft_model(model, cfg)\n        for p in model.parameters():\n            if p.requires_grad and p.dtype != torch.float32:\n                p.data = p.data.float()\n        R["loraTargetNames"] = sorted({n.split(".")[-1] for n, m in model.named_modules() if hasattr(m, "lora_A") and not isinstance(m, torch.nn.ModuleDict)})\n        tr = [p for p in model.parameters() if p.requires_grad]\n        R["trainableParams"] = sum(p.numel() for p in tr)\n        R["totalParams"] = sum(p.numel() for p in model.parameters())\n        R["trainableDtypes"] = sorted({str(p.dtype) for p in tr})\n        opt = torch.optim.AdamW(tr, lr=2e-4)\n    else:\n        import bitsandbytes as bnb\n        tr = [p for p in model.parameters() if p.requires_grad]\n        R["trainableParams"] = sum(p.numel() for p in tr)\n        opt = bnb.optim.AdamW8bit(tr, lr=1e-5)\n    model.train()\n    scaler = torch.amp.GradScaler("cuda")\n    torch.cuda.reset_peak_memory_stats()\n    micro = job["batch"]; times = []; losses = []; nonfinite = 0; skipped = 0; capped = False\n\n    def step_fn(batch, micro):\n        opt.zero_grad(set_to_none=True)\n        tot = 0.0\n        for i in range(0, batch.shape[0], micro):\n            x = batch[i:i + micro].cuda()\n            with torch.autocast("cuda", dtype=torch.float16):\n                loss = model(input_ids=x, labels=x).loss\n            frac = x.shape[0] / batch.shape[0]\n            tot += loss.detach().float().item() * frac\n            scaler.scale(loss * frac).backward()\n        prev = scaler.get_scale()\n        scaler.unscale_(opt)\n        torch.nn.utils.clip_grad_norm_(tr, 1.0)\n        scaler.step(opt); scaler.update()\n        return tot, scaler.get_scale() < prev\n\n    ts = time.time()\n    for s in range(N):\n        t1 = time.time()\n        while True:\n            try:\n                loss, sk = step_fn(stream[s], micro)\n                break\n            except torch.OutOfMemoryError:\n                R.setdefault("oomRetries", []).append({"step": s, "micro": micro})\n                opt.zero_grad(set_to_none=True); gc.collect(); torch.cuda.empty_cache()\n                if micro == 1:\n                    raise\n                micro //= 2\n        torch.cuda.synchronize()\n        dt = time.time() - t1\n        times.append(dt); losses.append(loss); skipped += int(sk)\n        if not (loss == loss and abs(loss) != float("inf")):\n            nonfinite += 1\n        print(f"step {s+1} loss {loss:.4f} {dt:.2f}s scale {scaler.get_scale()}", flush=True)\n        if time.time() - ts > job["time_cap"]:\n            capped = True\n            break\n    sel = times[5:] if len(times) > 6 else times[1:]\n    med = statistics.median(sel) if sel else None\n    R.update(stepsDone=len(times), microBatch=micro, timeCapHit=capped, tokensPerStep=B * T,\n             sPerStepMedian=round(med, 3) if med else None,\n             tokensPerSec=round(B * T / med, 1) if med else None,\n             stepTimes=[round(t, 2) for t in times], loss1=losses[0], lossLast=losses[-1], losses=[round(l, 4) for l in losses],\n             nonFiniteLosses=nonfinite, skippedSteps=skipped, gradScalerFinal=scaler.get_scale(),\n             peakMemGB=round(torch.cuda.max_memory_allocated() / 1e9, 2), ok=True)\n    R["warnings"] = logbuf.getvalue()[:2000]\nexcept Exception as e:\n    R["ok"] = False; R["error"] = f"{type(e).__name__}: {str(e)[:900]}"; R["trace"] = traceback.format_exc()[-1800:]\n    R["warnings"] = logbuf.getvalue()[:2000]\n    try:\n        R["peakMemGB"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)\n    except Exception:\n        pass\nsave()\n'
open("/kaggle/working/probe_worker.py", "w").write(WORKER)
import torch, transformers, peft
import bitsandbytes
try:
    import trl; trlv = trl.__version__
except Exception as e:
    trlv = "n/a"
ALL = {"date": time.strftime("%Y-%m-%d"), "gpu": torch.cuda.get_device_name(0), "gpuCount": torch.cuda.device_count(),
       "env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
               "transformers": transformers.__version__, "peft": peft.__version__, "bitsandbytes": bitsandbytes.__version__, "trl": trlv},
       "runs": []}
def fl():
    r = {}
    for m in ("fla", "causal_conv1d"):
        try:
            __import__(m); r[m] = True
        except Exception:
            r[m] = False
    return r
ALL["envFlaBefore"] = fl()
print(ALL, flush=True)
T0 = time.time()
def save():
    json.dump(ALL, open("/kaggle/working/llt_model_probe_9b.json", "w"), indent=1)
def job(tag, model, mode, steps, batch, cap):
    jf = f"/kaggle/working/job_{tag}.json"; rf = f"/kaggle/working/res_{tag}.json"
    json.dump(dict(tag=tag, model=model, mode=mode, steps=steps, batch=batch, time_cap=cap), open(jf, "w"))
    t = time.time()
    p = subprocess.run([sys.executable, "/kaggle/working/probe_worker.py", jf, rf], capture_output=True, text=True)
    tail = (p.stdout or "")[-1500:] + (p.stderr or "")[-1500:]
    try:
        r = json.load(open(rf))
    except Exception:
        r = {"tag": tag, "ok": False, "error": "no result file, worker crashed"}
    r["wallSec"] = round(time.time() - t, 1)
    r["tail"] = tail
    ALL["runs"].append(r); save()
    print(tag, {k: r.get(k) for k in ("ok", "loadedClass", "visionTowerLoaded", "sPerStepMedian", "peakMemGB", "loss1", "lossLast", "error")}, flush=True)
    print("elapsed min", round((time.time() - T0) / 60, 1), flush=True)


In [ ]:

job("q35-9b-qlora-v2", "Qwen/Qwen3.5-9B-Base", "qlora", 5, 2, 900)
